In [6]:
from m33_pipeline.config import get_derived_config
from m33_pipeline.derived import (
    add_clustering_metrics,
    add_electron_density,
    add_logU_KK04,
    add_metallicity_columns,
    add_metallicity_error_columns,
    add_symmetry_class,
    merge_field_flux_catalogs,
    write_clustering_outputs,
    write_total_flux_catalog,
)
from m33_pipeline.io import write_catalog
from m33_pipeline import paths
from m33_pipeline.validate import validate_total_catalog


# Merge per-field flux catalogs


In [7]:
derived_config = get_derived_config()
all_catalog = merge_field_flux_catalogs()
output_path = write_total_flux_catalog(all_catalog)
print("Combined catalog shape:", all_catalog.shape)
print("Saved combined catalog to:", output_path)
validate_total_catalog(all_catalog)


Combined catalog shape: (1189, 150)
Saved combined catalog to: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/total_flux_catalog.csv


[]

# Add ionization parameter


In [8]:
cat = add_logU_KK04(all_catalog.copy(), n_mc=derived_config.logu_n_mc, seed=123, metallicity_cal="M13_O3N2")
derived_output_path = paths.flux_catalog_dir() / "total_flux_catalog_with_derived.csv"
write_catalog(cat, derived_output_path)
print("Number of columns:", len(cat.columns))
print("Saved combined catalog with ionization parameter:", derived_output_path)


Number of columns: 165
Saved combined catalog with ionization parameter: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/total_flux_catalog_with_derived.csv


# Add electron density


In [9]:
# df = add_electron_density(cat.copy(), n_mc=derived_config.density_n_mc)
# derived_output_path = paths.flux_catalog_dir() / "total_flux_catalog_with_derived.csv"
# write_catalog(df, derived_output_path)
# print("Number of columns:", len(df.columns))
# print("Saved combined catalog with electron densities:", derived_output_path)


# Add symmetry classification


In [11]:
df = add_symmetry_class(cat.copy())
print("Symmetry classification counts:")
print(df["symmetry_class"].value_counts())
derived_output_path = paths.flux_catalog_dir() / "total_flux_catalog_with_derived_and_symmetry.csv"
write_catalog(df, derived_output_path)
print("Number of columns:", len(df.columns))
print("Saved combined catalog with symmetry classification:", derived_output_path)


Symmetry classification counts:
symmetry_class
asymmetric    1007
symmetric      182
Name: count, dtype: int64
Number of columns: 166
Saved combined catalog with symmetry classification: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/total_flux_catalog_with_derived_and_symmetry.csv


# Add metallicity calibrations


In [12]:
df = add_metallicity_columns(df.copy())
df = add_metallicity_error_columns(df.copy(), n_mc=derived_config.metallicity_n_mc, seed=123)
derived_output_path = paths.flux_catalog_dir() / "total_flux_catalog_with_derived_and_metallicities.csv"
write_catalog(df, derived_output_path)
print("Number of columns:", len(df.columns))
print("Saved combined catalog with metallicities:", derived_output_path)


/Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/m33_pipeline/derived.py:384: RuntimeWarning: invalid value encountered in log10
  out["Z_R_Pilyugin2016_highN2"] = 8.589 + 0.022 * np.log10(r3 / r2) + 0.399 * np.log10(n2) + (-0.137 + 0.164 * np.log10(r3 / r2) + 0.589 * np.log10(n2)) * np.log10(r2)
/Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/m33_pipeline/derived.py:385: RuntimeWarning: invalid value encountered in log10
  out["Z_R_Pilyugin2016_lowN2"] = 7.932 + 0.944 * np.log10(r3 / r2) + 0.695 * np.log10(n2) + (0.970 - 0.291 * np.log10(r3 / r2) - 0.019 * np.log10(n2)) * np.log10(r2)
/Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/m33_pipeline/derived.py:386: RuntimeWarning: invalid value encountered in log10
  out["Z_S_Pilyugin2016_highN2"] = 8.424 + 0.030 * np.log10(r3 / s2) + 0.751 * np.log10(n2) + (-0.349 + 0.182 * np.log10(r3 / s2) + 0.508 * np.log10(n2)) * np.log10(s2)
/Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/m33_pipeline/derived.py:387: RuntimeWarning: invalid value encounte

Number of columns: 218
Saved combined catalog with metallicities: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/total_flux_catalog_with_derived_and_metallicities.csv


# Add deprojected clustering metrics


In [13]:
clustered_df, global_stats, ripley_df, pcf_df = add_clustering_metrics(df.copy())
outputs = write_clustering_outputs(clustered_df, global_stats, ripley_df, pcf_df)
print("Saved catalog:", outputs["catalog"])
print("Saved global stats:", outputs["global"])
print("Saved Ripley profile:", outputs["ripley"])
print("Saved pair-correlation profile:", outputs["pcf"])


Saved catalog: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/total_flux_catalog_with_deprojected_clustering_metrics.csv
Saved global stats: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/clustering_global_statistics.csv
Saved Ripley profile: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/clustering_ripley_profile.csv
Saved pair-correlation profile: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/clustering_pair_correlation_profile.csv
